In [15]:
import numpy as np
import cupy as cp
import pandas as pd
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
from sklearn.neighbors import KNeighborsClassifier
import pprint
from sklearn.discriminant_analysis import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from parametrage import tune_and_evaluate 
import pandas as pd
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report 
from scratch import DecisionTree, KNN_GPU, df_to_gpu_array

## upload data 

In [16]:
 
data_dir = "Cleaned_dataset/Balanced_data/"
 
X_train = pd.read_csv(f"{data_dir}X_train_bal.csv")
Y_train = pd.read_csv(f"{data_dir}y_train_bal.csv")["fire_count"]  # ensure Series
 
X_test = pd.read_csv(f"{data_dir}X_test.csv")
Y_test = pd.read_csv(f"{data_dir}y_test.csv")["fire_count"]  # ensure Series

print("Loaded datasets successfully!")
print("X_train_bal:", X_train.shape)
print("y_train_bal:", Y_train.shape)
print("X_test:", X_test.shape)
print("y_test:", Y_test.shape)



Loaded datasets successfully!
X_train_bal: (158283, 39)
y_train_bal: (158283,)
X_test: (42298, 39)
y_test: (42298,)


In [17]:
train_df = pd.concat([X_train, Y_train], axis=1)
test_df  = pd.concat([X_test, Y_test], axis=1)

print(train_df.head)
print(test_df.head)

<bound method NDFrame.head of         spring_prec  autumn_prec  winter_prec  summer_prec  autumn_tmax  \
0         -0.529313    -0.770358    -0.619231    -0.823171     1.846154   
1         -0.529313    -0.770358    -0.619231    -0.823171     1.846154   
2         -0.529313    -0.770358    -0.619231    -0.823171     1.846154   
3         -0.529313    -0.770358    -0.619231    -0.823171     1.846154   
4         -0.482412    -0.749186    -0.596154    -0.829268     1.846154   
...             ...          ...          ...          ...          ...   
158278     2.732544     2.476407     1.765441    -0.038906    -0.656907   
158279     5.538422     4.044785     1.634335     0.643786    -1.523367   
158280     4.358241     1.536153     0.972058     0.487088    -1.344605   
158281     5.991171     5.135530     1.691755    -0.405621    -1.200792   
158282     0.925264     0.260769     0.286844     0.051985    -0.132580   

        elevation  autumn_tmin  summer_tmax  spring_tmax  winter_tmin

In [18]:
print(train_df['fire_count'].mean())
print(test_df['fire_count'].mean())



0.3052064972233278
0.16579980140905007


## Algorithme

In [19]:
knn_params = {
    "n_neighbors": [3, 5, 7, 9, 11],
}

dt_params = {
    "criterion": ["entropy"],
    "max_depth": [10, 20],
    "min_samples_split": [5, 10],
    "min_samples_leaf": [5, 10],
    }


In [20]:

print("\n================= 🌳 GRID SEARCH: Decision Tree =================\n")
dt_results = tune_and_evaluate(
    balanced_train_df=train_df,
    test_df=test_df,
    estimator=DecisionTreeClassifier(random_state=42),
    param_grid=dt_params,
    target_col="fire_count",
    cv_folds=5,
    scoring="recall",
    )



================= 🌳 GRID SEARCH: Decision Tree =================


...GridSearchCV...


Nolbres de combinations: 8

Combination Num_1/8
Params: {'criterion': 'entropy', 'max_depth': 10, 'min_samples_leaf': 5, 'min_samples_split': 5}
-----------------------------------


Metriques de la Cross-validation :
   • accuracy: 0.9750 
   • precision: 0.9973 
   • recall: 0.9207 
   • f1: 0.9575 
   • roc_auc: 0.9790
-----------------------------------

Combination Num_2/8
Params: {'criterion': 'entropy', 'max_depth': 10, 'min_samples_leaf': 5, 'min_samples_split': 10}
-----------------------------------
Metriques de la Cross-validation :
   • accuracy: 0.9750 
   • precision: 0.9973 
   • recall: 0.9207 
   • f1: 0.9575 
   • roc_auc: 0.9790
-----------------------------------

Combination Num_3/8
Params: {'criterion': 'entropy', 'max_depth': 10, 'min_samples_leaf': 10, 'min_samples_split': 5}
-----------------------------------
Metriques de la Cross-validation :
   • accuracy: 0.9748 
   • precision: 0.9965 
   • recall: 0.9206 
   • f1: 0.9571 
   • roc_auc: 0.9792
-----------------------------------

Combination Num_4/8
Params: {'criterion': 'entropy', 'max_depth': 10, 'min_samples_leaf': 10, 'min_samples_split': 10}
-----------------------------------
Met

In [21]:
print("\n================= 🔍 GRID SEARCH: KNN =================\n")
knn_results = tune_and_evaluate(
    balanced_train_df=train_df,
    test_df=test_df,
    estimator=KNeighborsClassifier(),
    param_grid=knn_params,
    target_col="fire_count",
    cv_folds=5,
    scoring="recall",
    )



================= 🔍 GRID SEARCH: KNN =================


...GridSearchCV...


Nolbres de combinations: 5

Combination Num_1/5
Params: {'n_neighbors': 3}
-----------------------------------
Metriques de la Cross-validation :
   • accuracy: 0.9773 
   • precision: 0.9833 
   • recall: 0.9417 
   • f1: 0.9621 
   • roc_auc: 0.9828
-----------------------------------

Combination Num_2/5
Params: {'n_neighbors': 5}
-----------------------------------
Metriques de la Cross-validation :
   • accuracy: 0.9748 
   • precision: 0.9884 
   • recall: 0.9285 
   • f1: 0.9575 
   • roc_auc: 0.9849
-----------------------------------

Combination Num_3/5
Params: {'n_neighbors': 7}
-----------------------------------
Metriques de la Cross-validation :
   • accuracy: 0.9731 
   • precision: 0.9927 
   • recall: 0.9185 
   • f1: 0.9542 
   • roc_auc: 0.9861
-----------------------------------

Combination Num_4/5
Params: {'n_neighbors': 9}
-----------------------------------
Metriques de la Cross-valid

# DECISION TREE FROM SCRATCH

In [22]:

tree = DecisionTree(
    criterion="entropy",
    max_depth=20,
    min_samples_split=5,
    min_samples_leaf=5
)

tree.fit(X_train, Y_train)
preds = tree.predict(X_test)

print("Accuracy:", accuracy_score(Y_test, preds))
print("Precision:", precision_score(Y_test, preds, zero_division=0))
print("Recall:", recall_score(Y_test, preds, zero_division=0))
print("F1:", f1_score(Y_test, preds, zero_division=0))
print(classification_report(Y_test, preds))
print("\nConfusion Matrix:")
print(confusion_matrix(Y_test, preds))

Accuracy: 0.9875880656295806
Precision: 0.9724730556364696
Recall: 0.9520889776130044
F1: 0.9621730672238634
              precision    recall  f1-score   support

         0.0       0.99      0.99      0.99     35285
         1.0       0.97      0.95      0.96      7013

    accuracy                           0.99     42298
   macro avg       0.98      0.97      0.98     42298
weighted avg       0.99      0.99      0.99     42298


Confusion Matrix:
[[35096   189]
 [  336  6677]]


# KNN  FROM SCRATCH

In [23]:
X_train_gpu = df_to_gpu_array(X_train)
X_test_gpu = df_to_gpu_array(X_test)

y_train_gpu = cp.asarray(Y_train.values.astype(np.int32))
Y_test = cp.asarray(Y_test.values.astype(np.int32))


tree = KNN_GPU  (
    k=3,
    task = "classification"
)

tree.fit(X_train_gpu, y_train_gpu)
preds = tree.predict(X_test_gpu).get()
Y_test = Y_test.get()

print("Accuracy:", accuracy_score(Y_test, preds))
print("Precision:", precision_score(Y_test, preds, zero_division=0))
print("Recall:", recall_score(Y_test, preds, zero_division=0))
print("F1:", f1_score(Y_test, preds, zero_division=0))
print(classification_report(Y_test, preds))
print("\nConfusion Matrix:")
print(confusion_matrix(Y_test, preds))

[KNN_GPU] OOM for batch_size; retrying with batch_size=2048
[KNN_GPU] OOM for batch_size; retrying with batch_size=1024
Accuracy: 0.9750815641401485
Precision: 0.9243697478991597
Recall: 0.9254242121773849
F1: 0.9248966794926606
              precision    recall  f1-score   support

           0       0.99      0.98      0.99     35285
           1       0.92      0.93      0.92      7013

    accuracy                           0.98     42298
   macro avg       0.95      0.96      0.95     42298
weighted avg       0.98      0.98      0.98     42298


Confusion Matrix:
[[34754   531]
 [  523  6490]]


In [ ]:
X_train_gpu = df_to_gpu_array(X_train)
X_test_gpu  = df_to_gpu_array(X_test)

y_train_gpu = cp.asarray(Y_train.astype(np.int32))
y_test_gpu  = cp.asarray(Y_test.astype(np.int32))

tree = KNN_GPU(
    k=3,
    task="classification"
)

tree.fit(X_train_gpu, y_train_gpu)

# Prédictions GPU → CPU
preds = tree.predict(X_test_gpu).get()
y_test = y_test_gpu.get()

print("Accuracy:", accuracy_score(y_test, preds))
print("Precision:", precision_score(y_test, preds, zero_division=0))
print("Recall:", recall_score(y_test, preds, zero_division=0))
print("F1:", f1_score(y_test, preds, zero_division=0))
print(classification_report(y_test, preds))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, preds))


[KNN_GPU] OOM for batch_size; retrying with batch_size=2048
[KNN_GPU] OOM for batch_size; retrying with batch_size=1024
Accuracy: 0.9775875927939855
Precision: 0.9667538864091119
Recall: 0.8956224155140453
F1: 0.9298297557364915
              precision    recall  f1-score   support

           0       0.98      0.99      0.99     35285
           1       0.97      0.90      0.93      7013

    accuracy                           0.98     42298
   macro avg       0.97      0.94      0.96     42298
weighted avg       0.98      0.98      0.98     42298


Confusion Matrix:
[[35069   216]
 [  732  6281]]
